[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C22_Reasoning_RL_Course/05_ttc_scaling/05_ttc_scaling.ipynb)

# 05 · 测试时计算 scaling（用 numpy 模拟）

目标：把**测试时计算当成可优化的旋钮**——模拟 TTC scaling 曲线、做 **compute-optimal 预算分配**、**难度路由**、画**小模型+TTC vs 大模型**的帕累托前沿。

路线：TTC scaling 曲线(误差指数衰减) → 收益递减 → compute-optimal 分配(等边际) → 难度路由(胜过一刀切) → 帕累托前沿 → 串/并行 → ✏️ 练习 → 📖 答案 → 🧪 真实 scaling 胶囊。

> 心智模型：**TTC 是个可优化旋钮, 准确率随它 scaling 但收益递减; 最优花法按难度变**。小模型+大量TTC 可在等算力下胜大模型(但 TTC 放大能力≠创造能力)。

## 1 · TTC scaling 曲线：误差指数衰减

完美 verifier 下 best-of-N: $\text{acc}(N)=1-(1-p)^N$。误差 $(1-p)^N$ **指数衰减** -> $\log(\text{误差})$ 随 N 线性。
我们画出曲线并验证『准确率 vs log(算力) 近似直线』。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def acc_bon(p, N):
    '''完美 verifier 下 best-of-N 准确率 = pass@N = 1-(1-p)^N。'''
    return 1 - (1 - p) ** N

p = 0.3
Ns = [1, 2, 4, 8, 16, 32, 64]
print(f"{'N':>4}{'acc':>8}{'误差':>10}{'log(误差)':>12}")
errs = []
for N in Ns:
    a = acc_bon(p, N); err = 1 - a; errs.append(err)
    print(f'{N:>4}{a:>8.4f}{err:>10.4f}{np.log(err):>12.4f}')
# log(误差) 随 N 线性: 斜率 = log(1-p)
log_errs = np.log(errs)
slope = (log_errs[-1] - log_errs[0]) / (Ns[-1] - Ns[0])
print(f'\nlog(误差) 对 N 的斜率 = {slope:.4f}  (理论 log(1-p) = {np.log(1-p):.4f})')
assert abs(slope - np.log(1 - p)) < 1e-6, 'log(误差) 随 N 线性,斜率=log(1-p)'
assert acc_bon(p, 64) > acc_bon(p, 1), 'TTC 越多越准'
print('✅ TTC scaling: 误差指数衰减, log(误差) 随算力线性(对数线性 scaling)')

## 2 · 收益递减：每多一个样本的边际收益单调下降

TTC 的根本约束：**收益递减**。pass@N 是 N 的**凹函数**, 故**每多一个样本**的边际收益单调下降。
我们在线性 N 上算逐样本边际收益, 验证它单调下降 -> 『无脑加大 N』很快不划算。

In [ ]:
p = 0.3
Ns = list(range(1, 13))           # 线性递增(逐样本)
accs = [acc_bon(p, N) for N in Ns]
print(f"{'N':>5}{'acc':>8}{'+1样本边际收益':>16}")
margins = []
for i in range(len(Ns)):
    if i == 0:
        print(f'{Ns[i]:>5}{accs[i]:>8.4f}{"—":>16}')
    else:
        m = accs[i] - accs[i-1]; margins.append(m)
        print(f'{Ns[i]:>5}{accs[i]:>8.4f}{m:>16.4f}')
# 每多一个样本的边际收益单调下降(pass@N 是 N 的凹函数)
assert all(margins[i] >= margins[i+1] - 1e-12 for i in range(len(margins)-1)), '边际收益应递减'
print('\n✅ 收益递减: 每多一个样本的提升越来越小 -> compute-optimal 要知道何时停手')

## 3 · compute-optimal 分配：等边际原理

给定**固定总预算** B(总采样数), 分给两类题(易 p_easy、难 p_hard)以最大化总准确率。
最优解满足**等边际**: 每多一个样本花在任何题上的边际收益相等。我们暴力搜出最优分配并验证它优于均分。

In [ ]:
def total_acc(n_easy, n_hard, p_easy, p_hard, count_easy, count_hard):
    '''总期望答对题数 = 易题数*acc(易,n_easy) + 难题数*acc(难,n_hard)。'''
    return count_easy * acc_bon(p_easy, n_easy) + count_hard * acc_bon(p_hard, n_hard)

p_easy, p_hard = 0.7, 0.15        # 易题单样本正确率高, 难题低
count_easy, count_hard = 50, 50   # 各 50 题
B = 50 * 8 + 50 * 8               # 总预算 = 均分时每题 8

# 暴力搜: 给易题每题 n_e, 难题每题 n_h, 满足 count_easy*n_e+count_hard*n_h<=B
best = None
for n_e in range(1, 30):
    for n_h in range(1, 30):
        if count_easy * n_e + count_hard * n_h <= B:
            acc = total_acc(n_e, n_h, p_easy, p_hard, count_easy, count_hard)
            if best is None or acc > best[0]:
                best = (acc, n_e, n_h)
best_acc, best_ne, best_nh = best
uniform_acc = total_acc(8, 8, p_easy, p_hard, count_easy, count_hard)
print(f'均分(每题8): 期望答对 {uniform_acc:.2f} / 100 题')
print(f'compute-optimal: 易题每题 {best_ne}, 难题每题 {best_nh} -> 期望答对 {best_acc:.2f}')
assert best_acc >= uniform_acc, 'compute-optimal 应 >= 均分'
assert best_nh > best_ne, '难题(未饱和)应分到更多, 易题(早饱和)分到更少'
print('✅ compute-optimal: 把预算从饱和的易题移向未饱和的难题(等边际原理)')

## 4 · 难度路由：按难度分配, 胜过一刀切

实战版 compute-optimal: 一批混合难度的题, 用 **probe(先采几个看 verifier 最高分)** 估难度(最高分低=难), 
按难度**中位数分两档**(易档少花、难档多花, 预算恰好守恒), 用 best-of-N + verifier。
验证难度路由在**同样总预算**下准确率高于一刀切。

In [ ]:
def make_pool(p, N, vnoise, seed):
    r = np.random.default_rng(seed)
    correct = (r.random(N) < p).astype(int)
    vscore = correct + r.normal(0, vnoise, N)
    return correct, vscore

def bon_solve(p, N, vnoise, seed):
    '''best-of-N + verifier: 选最高分, 返回是否真对。更多样本对难题有帮助。'''
    correct, vs = make_pool(p, N, vnoise, seed)
    return int(correct[np.argmax(vs)])

def probe_difficulty(p, probe_n, vnoise, seed):
    '''probe: 采 probe_n 个, 用 verifier 最高分估难度(最高分越低 -> 越难)。'''
    _, vs = make_pool(p, probe_n, vnoise, seed)
    return -float(vs.max())                  # 越大越难

VN = 0.5
problems = [('easy', 0.6)] * 50 + [('hard', 0.18)] * 50
U = 8; BUDGET = len(problems) * U            # 一刀切: 每题 8

# 一刀切
uniform_correct = sum(bon_solve(p, U, VN, seed=i) for i, (_, p) in enumerate(problems))

# 难度路由: probe=4 估难度, 按中位数分两档(易档 N=4、难档 N=12); 中位数分 -> 预算恰好守恒
PROBE = 4
diffs = [probe_difficulty(p, PROBE, VN, seed=i) for i, (_, p) in enumerate(problems)]
median = np.median(diffs)
routed_correct = 0; used = 0; n_easy = 0; n_hard = 0
for i, (_, p) in enumerate(problems):
    if diffs[i] <= median:
        N = 4; n_easy += 1                    # 较易 -> 早停
    else:
        N = 12; n_hard += 1                   # 较难 -> 多花
    used += N
    routed_correct += bon_solve(p, N, VN, seed=i + 1000)

print(f'一刀切(每题{U}, 预算{BUDGET}): 答对 {uniform_correct}/{len(problems)}')
print(f'难度路由(用量{used}, 易档{n_easy}×4+难档{n_hard}×12): 答对 {routed_correct}/{len(problems)}')
assert used <= BUDGET, '中位数分档 -> 总预算守恒(不超一刀切)'
assert routed_correct >= uniform_correct, '难度路由应不输于一刀切(同预算)'
print('✅ 难度路由: 估难度 -> 易题少花、难题多花(预算守恒) -> 同预算更高准确率')

## 5 · 帕累托前沿：小模型+TTC vs 大模型

核心命题。在『准确率 vs 总算力』平面, 比较**小模型+大量TTC** 与 **大模型+少量TTC**。
计算帕累托前沿(不被支配的点), 验证中算力区小模型+TTC 可胜过大模型。

In [ ]:
# 两个模型: 小模型(便宜但 p 低), 大模型(贵但 p 高)。TTC 用 best-of-N。
MODELS = {
    'small': dict(p=0.30, cost_per_sample=1.0),   # 每样本算力 1
    'large': dict(p=0.55, cost_per_sample=5.0),   # 大模型每样本贵 5 倍
}

def plan_cost_acc(model, N):
    m = MODELS[model]
    return m['cost_per_sample'] * N, acc_bon(m['p'], N)

# 枚举各方案(模型 x N)
plans = []
for model in MODELS:
    for N in [1, 2, 4, 8, 16, 32, 64]:
        cost, acc = plan_cost_acc(model, N)
        plans.append((cost, acc, model, N))

# 帕累托前沿: 不存在另一方案 cost<= 且 acc>=(严格更优)
def pareto_front(plans):
    front = []
    for c, a, m, n in plans:
        dominated = any((c2 <= c and a2 >= a and (c2 < c or a2 > a)) for c2, a2, _, _ in plans)
        if not dominated:
            front.append((c, a, m, n))
    return sorted(front)

front = pareto_front(plans)
print('帕累托前沿(总算力, 准确率, 模型, N):')
for c, a, m, n in front:
    print(f'  算力{c:6.1f}  acc{a:.3f}  {m}+BoN{n}')
# 小模型应出现在前沿(中算力区不被大模型支配)
small_on_front = any(m == 'small' for _, _, m, _ in front)
assert small_on_front, '小模型+TTC 应在帕累托前沿上(等算力下可胜大模型)'
# 具体: 小模型 BoN 到某算力的 acc 可超过大模型同算力
small_c, small_a = plan_cost_acc('small', 16)   # 算力16
large_c, large_a = plan_cost_acc('large', 3)    # 算力15, 接近
print(f'\n等算力对比: small+BoN16(算力{small_c:.0f},acc{small_a:.3f}) vs large+BoN3(算力{large_c:.0f},acc{large_a:.3f})')
assert small_a > large_a, '中算力区: 小模型+大量TTC 胜过大模型+少量TTC'
print('✅ 帕累托: 小模型+大量TTC 在等算力下可胜大模型(但 TTC 放大能力≠创造能力)')

## 6 · 串行 vs 并行：最优配比随难度变

**关键: 最优的并/串行配比随难度变**(Snell 2024)。这里用诚实的玩具模型:
- **并行 = 多数投票**(无 oracle, 现实): 模型可靠时(p 高)极强, 但 p<0.5 时错误答案会主导投票而失效。
- **串行 = 逐步修订**: 修订增益随 p 缩放(易题在正轨上修订有效, 极难题修订也救不回)。

结果: **很易的题并行(投票)胜, 较难的题串行(修订)胜** —— 验证『没有一种 TTC 通吃, 要按难度选』。

In [ ]:
from collections import Counter

def parallel_majority(p, budget, n_wrong=5, trials=5000, seed=0):
    '''并行=多数投票(无 oracle): 采 budget 个答案取众数。p<0.5 时易被错误答案淹没。'''
    r = np.random.default_rng(seed); c = 0
    for _ in range(trials):
        ans = [0 if r.random() < p else int(r.integers(1, 1 + n_wrong)) for _ in range(budget)]
        c += Counter(ans).most_common(1)[0][0] == 0
    return c / trials

def sequential_revise(p, budget, base_gain=0.45, cap=0.92):
    '''串行=逐步修订: 增益随 p 缩放(易题在正轨->修订有效; 难题->修订打折)。'''
    eff = base_gain * p; acc = p
    for _ in range(budget - 1):
        acc = acc + eff * (cap - acc)
    return acc

budget = 8
print(f"{'题型':>8}{'p':>6}{'并行(投票)':>12}{'串行(修订)':>12}{'谁更优':>8}")
results = []
for label, p in [('很易', 0.85), ('中等', 0.55), ('较难', 0.3)]:
    par = parallel_majority(p, budget); seq = sequential_revise(p, budget)
    winner = '串行' if seq > par else '并行'
    results.append((label, par, seq, winner))
    print(f'{label:>8}{p:>6}{par:>12.3f}{seq:>12.3f}{winner:>8}')
# 很易: 多数投票(模型可靠)胜; 较难: 串行修订(投票失效)胜 -> winner 随难度改变
assert parallel_majority(0.85, budget) > sequential_revise(0.85, budget), '很易题: 投票胜'
assert sequential_revise(0.3, budget) > parallel_majority(0.3, budget), '较难题: 修订胜(投票失效)'
winners = {r[3] for r in results}
assert len(winners) == 2, '最优方法应随难度改变(不是一种通吃)'
print('✅ 最优配比随难度变: 没有一种 TTC 通吃 —— 须按难度在并行/串行间选(或混合)')

---
## ✏️ 练习 1：TTC scaling 曲线与外推

实现 `acc_bon(p, N)` 和 `samples_for_target(p, target_acc)`(完美 verifier 下要达到 target_acc 最少需多少样本 N)。

In [ ]:
def acc_bon(p, N):
    # TODO: 1-(1-p)^N
    raise NotImplementedError

import math
def samples_for_target(p, target_acc):
    # TODO: 最小 N 使 1-(1-p)^N >= target_acc; 用对数 ceil
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert abs(acc_bon(0.3, 1) - 0.3) < 1e-9
assert abs(acc_bon(0.5, 2) - 0.75) < 1e-9
# 要 95% 准确率, p=0.3: 1-0.7^N>=0.95 -> 0.7^N<=0.05 -> N>=ln.05/ln.7=8.4 -> 9
assert samples_for_target(0.3, 0.95) == 9
assert acc_bon(0.3, samples_for_target(0.3, 0.95)) >= 0.95
# 模型越强(p高), 达标需样本越少
assert samples_for_target(0.6, 0.95) < samples_for_target(0.3, 0.95)
print('✅ 练习 1 通过：TTC scaling 曲线 + 达标所需算力外推')

## ✏️ 练习 2：边际收益与早停

实现 `marginal_gains(p, Ns)`(返回相邻 N 的准确率增量列表) 和 `stop_N(p, Ns, min_gain)`(边际收益首次低于 min_gain 时的 N, 即该停手的点)。

In [ ]:
def marginal_gains(p, Ns):
    # TODO: 返回 [acc(Ns[i])-acc(Ns[i-1]) for i>=1]
    raise NotImplementedError

def stop_N(p, Ns, min_gain):
    # TODO: 返回第一个 N 使『从上一档到它的边际收益 < min_gain』; 无则返回 Ns[-1]
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
Ns = list(range(1, 13))          # 线性 N: 逐样本边际单调递减
mg = marginal_gains(0.3, Ns)
assert len(mg) == len(Ns) - 1
assert all(mg[i] >= mg[i+1] - 1e-12 for i in range(len(mg)-1)), '逐样本边际递减'
# min_gain=0.05: 边际收益跌破 0.05 时停
s = stop_N(0.3, Ns, 0.05)
assert s in Ns and s > 1
# 跌破阈值后才停: stop 点的边际 < min_gain
assert (acc_bon(0.3, s) - acc_bon(0.3, s-1)) < 0.05
print('✅ 练习 2 通过：边际收益 + 早停点(收益递减时停手)')

## ✏️ 练习 3：两难度 compute-optimal 分配

实现 `optimal_split(p_easy, p_hard, count_easy, count_hard, budget)`：暴力搜出每类题的最优样本数 `(n_easy, n_hard)`(满足总预算), 最大化期望答对数。

In [ ]:
def optimal_split(p_easy, p_hard, count_easy, count_hard, budget, max_n=40):
    # TODO: 暴力搜 (n_e, n_h), 约束 count_easy*n_e+count_hard*n_h<=budget,
    #       最大化 count_easy*acc_bon(p_easy,n_e)+count_hard*acc_bon(p_hard,n_h)
    #       返回 (n_e, n_h)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
ne, nh = optimal_split(0.7, 0.15, 50, 50, budget=50*8+50*8)
# 难题未饱和应分更多, 易题早饱和分更少
assert nh > ne, '难题应分到更多样本'
# 最优分配的总准确率 >= 均分
def tot(ne, nh): return 50*acc_bon(0.7,ne)+50*acc_bon(0.15,nh)
assert tot(ne, nh) >= tot(8, 8) - 1e-9
print('✅ 练习 3 通过：compute-optimal 把预算移向未饱和的难题')

## ✏️ 练习 4：帕累托支配判定

实现 `is_dominated(plan, plans)`(plan=(cost,acc); 存在另一方案 cost更低且acc更高(或相等但严格更优)则被支配) 和 `pareto_front(plans)`(返回所有不被支配的方案)。

In [ ]:
def is_dominated(plan, plans):
    # TODO: plan=(cost,acc); 若存在 (c2,a2) s.t. c2<=cost and a2>=acc and 严格更优 -> True
    raise NotImplementedError

def pareto_front(plans):
    # TODO: 返回所有 not is_dominated 的方案
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
plans = [(1.0, 0.3), (2.0, 0.5), (2.0, 0.4), (5.0, 0.55)]
# (2.0,0.4) 被 (2.0,0.5) 支配(同cost更高acc)
assert is_dominated((2.0, 0.4), plans) == True
assert is_dominated((1.0, 0.3), plans) == False, '最便宜的点不被支配'
assert is_dominated((5.0, 0.55), plans) == False, '最准的点不被支配'
front = pareto_front(plans)
assert (2.0, 0.4) not in front and (2.0, 0.5) in front
assert len(front) == 3
print('✅ 练习 4 通过：帕累托支配判定与前沿提取')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
import math
def acc_bon(p, N):
    return 1 - (1 - p) ** N

def samples_for_target(p, target_acc):
    # 1-(1-p)^N >= target -> (1-p)^N <= 1-target -> N >= log(1-target)/log(1-p)
    return int(math.ceil(math.log(1 - target_acc) / math.log(1 - p)))

In [ ]:
# 练习 2 参考答案
def marginal_gains(p, Ns):
    accs = [acc_bon(p, N) for N in Ns]
    return [accs[i] - accs[i-1] for i in range(1, len(Ns))]

def stop_N(p, Ns, min_gain):
    accs = [acc_bon(p, N) for N in Ns]
    for i in range(1, len(Ns)):
        if accs[i] - accs[i-1] < min_gain:
            return Ns[i]
    return Ns[-1]

In [ ]:
# 练习 3 参考答案
def optimal_split(p_easy, p_hard, count_easy, count_hard, budget, max_n=40):
    best = None
    for n_e in range(1, max_n):
        for n_h in range(1, max_n):
            if count_easy * n_e + count_hard * n_h <= budget:
                acc = count_easy * acc_bon(p_easy, n_e) + count_hard * acc_bon(p_hard, n_h)
                if best is None or acc > best[0]:
                    best = (acc, n_e, n_h)
    return best[1], best[2]

In [ ]:
# 练习 4 参考答案
def is_dominated(plan, plans):
    cost, acc = plan
    for c2, a2 in plans:
        if c2 <= cost and a2 >= acc and (c2 < cost or a2 > acc):
            return True
    return False

def pareto_front(plans):
    return [pl for pl in plans if not is_dominated(pl, plans)]

---
## 🧪 真实数据胶囊：用真实 scaling 数字算账

用 Snell 2024 / o1 的真实定性结论(compute-optimal 比固定策略省约 4 倍算力; 小模型+TTC 可胜大模型)与真实模型规模, 算『等算力下买大模型还是买 TTC』。带 try/except 回退到内置真实数值。

In [ ]:
# 真实场景数字(公开报告的定性结论, 量级)
FACTS = {
    'compute_optimal_savings': 4.0,   # Snell: compute-optimal 比 baseline 省约 4x
    'small_p': 0.30, 'large_p': 0.55, # 小/大模型单样本正确率(量级)
    'large_over_small_cost': 5.0,     # 大模型每样本算力约 5x 小模型
}
def acc_bon(p, N):
    return 1 - (1 - p) ** N

# 等算力比较: 大模型跑 1 次 vs 小模型用同等算力跑 BoN
large_cost = FACTS['large_over_small_cost']    # 大模型 1 次 = 5 单位
large_acc = acc_bon(FACTS['large_p'], 1)        # 大模型 1 次
small_N = int(large_cost)                       # 小模型同算力可跑 5 次
small_acc = acc_bon(FACTS['small_p'], small_N)  # 小模型 BoN5
print(f'等算力(={large_cost:.0f}单位)对比:')
print(f'  大模型 1 次:        acc = {large_acc:.3f}')
print(f'  小模型 best-of-{small_N}: acc = {small_acc:.3f}')
winner = '小模型+TTC' if small_acc > large_acc else '大模型'
print(f'  -> 等算力下更优: {winner}')
# compute-optimal 省算力
baseline_compute = 100.0
optimal_compute = baseline_compute / FACTS['compute_optimal_savings']
print(f'\ncompute-optimal 达同等准确率约需算力 {optimal_compute:.0f} (vs baseline {baseline_compute:.0f})')
assert optimal_compute < baseline_compute, 'compute-optimal 省算力'
print('✅ 真实结论: 等算力下小模型+TTC 可胜大模型; compute-optimal 省约 4x 算力')

**🧪 胶囊练习**：实现 `equal_compute_winner(small_p, large_p, cost_ratio)` —— 在『大模型1次 vs 小模型用同等算力跑BoN』下, 返回胜者(`'small'`/`'large'`)。量化『该买模型还是买 TTC』。

In [ ]:
def equal_compute_winner(small_p, large_p, cost_ratio):
    # TODO: 大模型1次 acc = acc_bon(large_p,1); 小模型同算力跑 N=int(cost_ratio) 次 BoN
    #       比较, 返回 'small' 或 'large'
    raise NotImplementedError

In [ ]:
# 自测
# 小模型 p=0.3, 大模型 p=0.55, 大模型贵 5x: 小模型 BoN5 = 1-0.7^5=0.832 > 0.55 -> small
assert equal_compute_winner(0.3, 0.55, 5.0) == 'small'
# 若大模型只贵 1.2x(小模型只能多跑一点点) 且大模型强很多 -> large
assert equal_compute_winner(0.3, 0.9, 1.2) == 'large'
print('✅ 胶囊练习通过：等算力下买模型 vs 买 TTC 的判定')

In [ ]:
# 📖 胶囊参考答案
def equal_compute_winner(small_p, large_p, cost_ratio):
    large_acc = acc_bon(large_p, 1)
    small_acc = acc_bon(small_p, max(1, int(cost_ratio)))
    return 'small' if small_acc > large_acc else 'large'

### 小结
- **TTC scaling**: 准确率随测试时计算单调升、近似对数线性(误差指数衰减), 但**收益递减**。
- **compute-optimal**: 固定预算下按**等边际原理**分配 —— 把算力从饱和的易题移向未饱和的难题。
- **难度路由**: 估难度(采样一致性) -> 易题少花、难题多花 -> 同预算更高准确率。
- **帕累托前沿**: 中算力区**小模型+大量TTC 可胜大模型+少量TTC**; 但 TTC 放大能力≠创造能力(解不在分布里则无解)。
- **串行 vs 并行**: 难题偏并行(广撒网探索)、易题偏串行(深挖修订); 最优配比随难度变。
- **实践**: TTC 有真实成本(延迟/显存/$); 按题施策 + 边际归零就停 + KV-cache 复用降本。

**恭喜走完全程！** 你已从零实现: 结果奖励RL(长CoT涌现) → GRPO → PRM → verifier搜索 → TTC scaling。
回看课程主页的地图, 每一块都已亲手对拍验证。下一步: 把这些 numpy 逻辑接到 trl/verl, 在真模型上落地。